In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 1
max_iters = 1000
learning_rate = 3e-4
eval_iters = 250

cpu


In [3]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocabulary_size = len(chars)

In [4]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([92, 49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72,
        60, 63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73,
        78, 66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,
         1, 67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63,
        31, 73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77,
        63,  1, 73, 64,  1, 59, 72, 83, 73, 72])


In [5]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x.shape)
print(x)
print('targets')
print(y)

inputs:
torch.Size([1, 8])
tensor([[15,  1, 31, 79, 78,  1, 78, 66]])
targets
tensor([[ 1, 31, 79, 78,  1, 78, 66, 63]])


In [6]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, "target is", target)


when input is tensor([92]) target is tensor(49)
when input is tensor([92, 49]) target is tensor(66)
when input is tensor([92, 49, 66]) target is tensor(63)
when input is tensor([92, 49, 66, 63]) target is tensor(1)
when input is tensor([92, 49, 66, 63,  1]) target is tensor(45)
when input is tensor([92, 49, 66, 63,  1, 45]) target is tensor(76)
when input is tensor([92, 49, 66, 63,  1, 45, 76]) target is tensor(73)
when input is tensor([92, 49, 66, 63,  1, 45, 76, 73]) target is tensor(68)


In [7]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = loss.mean()
        model.train()
        return out

In [8]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)

    def forward(self, index, targets=None):

        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self.forward(index)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            index_next = torch.multinomial(probs, num_samples=1)

            index = torch.cat((index, index_next), dim=1)

        return index


model = BigramLanguageModel(vocabulary_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated_chars = decode(
    m.generate(context, max_new_tokens=500)[0].tolist()
)


    
print(context)
print(target)
print(int_to_string[context.item()])
print(int_to_string[target.item()])


first_letter = generated_chars[1]
print(first_letter)

token = string_to_int[first_letter]
print(token)


print(f" Generated Chars.start {generated_chars}")





tensor([[0]])
tensor(68)


j
•
90
 Generated Chars.start 
•y5?U3OwGL%SAR—”+”Un1(Cof:i8rkdyd8F&#E 
+$l.™FeDD8oCY0uEm8Dn6S_jQRG,lZB
G,O!U++xOB“mVkETD(C/x( dWBTvN],EE-Y%(3y7jeakvWAC2m
6c%7?(M_O)Hj2e‘w]L&,2OF[rOb5WWKr!pS7Gccdv4Jg :)r5J &/BKS﻿Hi1$•bqJc‘X“Qg[ufuycXo0+jFS:ZC.vFo9‘pM(CA1h﻿pERz?GhTJ8IG"&4X™RmfF_t'HWN'$MxLy.”7'A'jjv"S_&)]T;5f_/b]Hw0",)-4TALwC(3')A/]:‘8•1D t7Tc%9?/AO!/ejR﻿#Wo9[YUnX)-/kBg'﻿(m;p •,)jwl—x•,:X[+gGr8N"Bg1DA/wUSU5p$Gr2qDRKQF™6aM*P3﻿A/aoVL•i0qD[+E0e™7™TL—X?ANksjvimevW’q2k(mR]”z•/.Z’UzFNJZJ'I'_™
+‘KZAxYHp+?Me”PFylACT”Nufl641!/e(7“﻿gtQl—+


In [11]:


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    losses = estimate_loss()
    if iter % eval_iters == 0:
        print(f"step: {iter}, train: {losses['train']}")
    
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step: 0, train: 4.531933784484863
step: 250, train: 4.272374629974365
step: 500, train: 4.30396032333374
step: 750, train: 4.3667168617248535
step: 1000, train: 4.667762279510498
step: 1250, train: 4.305374622344971
step: 1500, train: 4.720224857330322
step: 1750, train: 4.074129581451416
step: 2000, train: 3.559173345565796
step: 2250, train: 4.197353363037109
step: 2500, train: 4.66277551651001
step: 2750, train: 4.141544342041016
step: 3000, train: 5.007401466369629
step: 3250, train: 3.434823513031006
step: 3500, train: 4.110044956207275
step: 3750, train: 4.543129920959473
step: 4000, train: 4.10167932510376
step: 4250, train: 4.255546569824219
step: 4500, train: 4.397391319274902
step: 4750, train: 3.210486650466919
step: 5000, train: 3.674865245819092
step: 5250, train: 3.9426567554473877
step: 5500, train: 4.504471302032471
step: 5750, train: 3.488732099533081
step: 6000, train: 3.891083002090454
step: 6250, train: 3.640482187271118
step: 6500, train: 3.330253839492798
step: 67

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)